# Monte Carlo Validation

Validate the Metropolis-Hastings sampler for scalar φ⁴ theory against known results:
1. Free field (λ=0): Gaussian distribution with known variance
2. Phase transition: order parameter vs coupling
3. Two-point correlation function decay

In [ ]:
import os, sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/qft_graph'
    !pip install -q torch-geometric omegaconf
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)
else:
    sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt

from qft_graph.config import LatticeConfig, ScalarFieldConfig, MCConfig
from qft_graph.lattice.hypercubic import HypercubicLattice
from qft_graph.actions.phi4 import Phi4Action
from qft_graph.mc.metropolis import MetropolisSampler
from qft_graph.mc.observables import ObservableSet
from qft_graph.mc.analysis import jackknife_mean_error
from qft_graph.utils.reproducibility import set_seed

set_seed(42)
plt.style.use('dark_background')
%matplotlib inline

## 1. Free Field Validation (λ=0)

In [ ]:
lattice = HypercubicLattice(LatticeConfig(dimensions=(16, 16)))
action = Phi4Action(lattice, ScalarFieldConfig(mass_squared=1.0, coupling=0.0))
sampler = MetropolisSampler(action, MCConfig(n_configs=2000, n_thermalization=500, n_sweeps_between=10, step_size=0.7, seed=42))

result = sampler.generate(2000)
print(f'Acceptance rate: {result.acceptance_rate:.3f}')
print(f'Mean action: {result.actions.mean():.2f} +/- {result.actions.std():.2f}')

# Histogram of field values (should be Gaussian)
fig, ax = plt.subplots(figsize=(8, 5))
all_vals = result.configurations.flatten().numpy()
ax.hist(all_vals, bins=100, density=True, alpha=0.7, label='MC samples')
x = np.linspace(-3, 3, 200)
ax.set_xlabel(r'$\phi$')
ax.set_ylabel('Density')
ax.set_title(r'Free field ($\lambda=0$, $m^2=1$): should be Gaussian')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Phase Transition: Order Parameter vs m²

In [ ]:
m2_values = np.linspace(-1.0, 0.0, 15)
mags = []
mag_errs = []

for m2 in m2_values:
    action = Phi4Action(lattice, ScalarFieldConfig(mass_squared=m2, coupling=0.5))
    sampler = MetropolisSampler(action, MCConfig(n_configs=500, n_thermalization=300, n_sweeps_between=5, seed=42))
    result = sampler.generate(500)
    obs = ObservableSet(lattice)
    m_samples = torch.tensor([obs.abs_magnetization(result.configurations[i]) for i in range(500)])
    mean, err = jackknife_mean_error(m_samples)
    mags.append(mean)
    mag_errs.append(err)
    print(f'm²={m2:.2f}: |M|={mean:.4f} ± {err:.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(m2_values, mags, yerr=mag_errs, fmt='o-', capsize=3)
ax.set_xlabel(r'$m^2$')
ax.set_ylabel(r'$|\langle\phi\rangle|$')
ax.set_title(r'Order parameter vs $m^2$ ($\lambda=0.5$, $L=16$)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Two-Point Correlation Function

In [ ]:
# Near critical point
action = Phi4Action(lattice, ScalarFieldConfig(mass_squared=-0.5, coupling=0.5))
sampler = MetropolisSampler(action, MCConfig(n_configs=1000, n_thermalization=500, n_sweeps_between=10, seed=42))
result = sampler.generate(1000)
obs = ObservableSet(lattice)

# Average G(r) over configurations
G_r_avg = torch.zeros(lattice.shape[0] // 2 + 1)
for i in range(min(200, len(result.configurations))):
    G_r_avg += obs.two_point_function(result.configurations[i])
G_r_avg /= 200

xi = ObservableSet.correlation_length(G_r_avg)
print(f'Correlation length: xi = {xi:.3f}')
print(f'xi/L = {xi/16:.4f}')

fig, ax = plt.subplots(figsize=(8, 5))
r = np.arange(len(G_r_avg))
ax.semilogy(r, G_r_avg.abs().numpy(), 'o-', label=r'$|G(r)|$')
if xi > 0 and xi < 100:
    ax.semilogy(r[1:], G_r_avg[1].item() * np.exp(-r[1:]/xi), '--', label=rf'$\xi={xi:.2f}$', alpha=0.7)
ax.set_xlabel(r'$r/a$')
ax.set_ylabel(r'$|G(r)|$')
ax.set_title(r'Two-point function ($m^2=-0.5$, $\lambda=0.5$)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()